<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 02b — Model Registry & Deployment

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~0h45

</div>

### Anaïs DUC-MARTIN

## 📋 Lab Overview

Once a model is trained and tracked, the next step in MLOps is to **register** it in a central catalog and **deploy** it behind an endpoint so it can serve predictions. The **Vertex AI Model Registry** provides versioning, aliasing (development / staging / production), and one-click deployment — all managed by GCP.

In this lab you will use two pre-trained TensorFlow image-classification models (ResNet V2) to practice the full lifecycle: upload, version, deploy, query, undeploy, and delete.

### Learning Objectives

1. Upload a model to the Vertex AI Model Registry.
2. Upload a second version of the same model and manage versioning.
3. Change the default version and assign aliases.
4. Create an endpoint and deploy a model to it.
5. Inspect deployment metadata and undeploy.
6. Delete model versions and clean up resources.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install dependencies, imports, GCP configuration |
| 1 | Prepare Models | Download two pre-trained ResNet models and save artifacts |
| 2 | Model Registry | Upload v1 and v2, list versions, manage aliases |
| 3 | Endpoints | Create an endpoint and deploy a model version |
| 4 | Manage Versions | Change default, delete a version |
| 5 | Cleanup | Undeploy, delete endpoint and models |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [1]:
%pip install --upgrade --quiet google-cloud-aiplatform

### 0.2 Imports

In [2]:
import os

import google.cloud.aiplatform as aip
import tensorflow as tf
import tensorflow_hub as hub
import tf_keras

print(f"TensorFlow version: {tf.__version__}")
print(f"Vertex AI SDK version: {aip.__version__}")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow version: 2.19.0
Vertex AI SDK version: 1.140.0


### 0.3 Configuration

In [3]:
# ── Constants ──
PROJECT_ID = "isae-sdd-481407"          # @param {type:"string"}
LOCATION = "europe-west3"                # @param {type:"string"}
BUCKET_URI = "gs://bucket-lab02"  # @param {type:"string"}

##############################  TODO  ##############################
# Set your_name to a unique lowercase identifier (e.g. first letter of first name + last name).
your_name = 'adm'  # TODO
####################################################################

aip.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}")

✅ Vertex AI initialized — project=isae-sdd-481407


---
## 1 · Prepare Models

We will use two pre-trained image-classification models from TensorFlow Hub. They differ in architecture size and input resolution, simulating a common real-world scenario where you iterate on model variants.

| Version | Model | Input size | Parameters |
|---------|-------|-----------|------------|
| v1 | ResNet V2 **101** | 224×224 | ~44M |
| v2 | ResNet V2 **50** | 128×128 | ~25M |

> 📖 **Docs:** [TensorFlow Hub models](https://tfhub.dev/s?module-type=image-classification)

In [4]:
# Download and build model v1 (ResNet V2 101)
tfhub_model_v1 = tf_keras.Sequential(
    [hub.KerasLayer("https://tfhub.dev/google/imagenet/resnet_v2_101/classification/5")]
)
tfhub_model_v1.build([None, 224, 224, 3])
tfhub_model_v1.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 1001)              44677609  
                                                                 
Total params: 44677609 (170.43 MB)
Trainable params: 0 (0.00 Byte)
Non-trainable params: 44677609 (170.43 MB)
_________________________________________________________________


In [5]:
# Download and build model v2 (ResNet V2 50)
tfhub_model_v2 = tf_keras.Sequential(
    [hub.KerasLayer("https://www.kaggle.com/models/google/resnet-v2/TensorFlow2/50-classification/2")]
)
tfhub_model_v2.build([None, 128, 128, 3])
tfhub_model_v2.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer_1 (KerasLayer)  (None, 1001)              25615849  
                                                                 
Total params: 25615849 (97.72 MB)
Trainable params: 0 (0.00 Byte)
Non-trainable params: 25615849 (97.72 MB)
_________________________________________________________________


**✏️ Question 1 — Model comparison**

a) What are the key differences between v1 (ResNet 101) and v2 (ResNet 50) in terms of size and expected behavior?

b) In what scenario would you prefer the smaller model?

---
*Your answer:*

a) ResNet 101 is about twice as big as ResNet 50 (since it has about twice as many parameters). ResNet 101 should have a better performance.

b) The smaller model can be used when the user has storage constraints or when the task can tolerate some errors.





---

### 1.1 Save model artifacts to Cloud Storage

Before uploading to the registry, the model artifacts must be stored in a GCS bucket.

In [6]:
MODEL_V1_DIR = f"{BUCKET_URI}/{your_name}/model/v1"
tfhub_model_v1.save(MODEL_V1_DIR)
print(f"✅ Model v1 saved to {MODEL_V1_DIR}")

MODEL_V2_DIR = f"{BUCKET_URI}/{your_name}/model/v2"
tfhub_model_v2.save(MODEL_V2_DIR)
print(f"✅ Model v2 saved to {MODEL_V2_DIR}")

✅ Model v1 saved to gs://bucket-lab02/adm/model/v1
✅ Model v2 saved to gs://bucket-lab02/adm/model/v2


### 1.2 Define the deployment container image

Vertex AI uses pre-built Docker containers to serve models. We select a TensorFlow CPU container.

> 📖 **Docs:** [Pre-built prediction containers](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers)

In [7]:
TF_VERSION = "2.13".replace(".", "-")
DEPLOY_VERSION = f"tf2-cpu.{TF_VERSION}"
DEPLOY_IMAGE = f"{LOCATION.split('-')[0]}-docker.pkg.dev/vertex-ai/prediction/{DEPLOY_VERSION}:latest"

print(f"Container image: {DEPLOY_IMAGE}")

Container image: europe-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-13:latest


---
## 2 · Model Registry

The Vertex AI Model Registry lets you group multiple versions of a model under a single resource. Key capabilities include: tracking model lineage across versions, assigning aliases (e.g. `production`, `development`), setting a default version for deployment, and managing the full lifecycle.

> 📖 **Docs:** [Model Registry overview](https://cloud.google.com/vertex-ai/docs/model-registry/introduction)

### 2.1 Upload version 1

In [8]:
MODEL_DISPLAY_NAME = f"lab02b-resnet-{your_name}"

model_v1 = aip.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    artifact_uri=MODEL_V1_DIR,
    serving_container_image_uri=DEPLOY_IMAGE,
    is_default_version=True,
    version_aliases=["v1"],
    version_description="ResNet V2 101 — baseline, 224x224 input",
)

print(f"✅ Model v1 uploaded: {model_v1.resource_name}")

✅ Model v1 uploaded: projects/588027604890/locations/europe-west3/models/1342622444770295808


> 💡 **Tip:** Go to the Vertex AI console → Model Registry to verify that the model appears with version ID 1.

### 2.2 Upload version 2

To add a new version to an existing model, use the `parent_model` parameter to link it to the first version.

In [11]:
##############################  TODO  ##############################
# Upload model v2 as a new version of the same model.
# Use the resource_name of the first version to link them.
# Set is_default_version=True so v2 becomes the new default.
# Assign the alias "v2" and add a brief description.

model_v2 = aip.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    artifact_uri=MODEL_V2_DIR,                        # TODO
    serving_container_image_uri=DEPLOY_IMAGE,
    parent_model='1342622444770295808',                        # TODO
    is_default_version=True,                  # TODO
    version_aliases=['v2'],                   # TODO
    version_description="ResNet V2 50 — baseline, 128x128 input",                 # TODO
)
####################################################################

print(f"✅ Model v2 uploaded: {model_v2.resource_name}")

✅ Model v2 uploaded: projects/588027604890/locations/europe-west3/models/1342622444770295808


### 2.3 List all versions

In [12]:
versions = model_v1.versioning_registry.list_versions()
for v in versions:
    print(v)

VersionInfo(version_id='1', version_create_time=DatetimeWithNanoseconds(2026, 3, 9, 14, 5, 13, 464990, tzinfo=datetime.timezone.utc), version_update_time=DatetimeWithNanoseconds(2026, 3, 9, 14, 12, 6, 218808, tzinfo=datetime.timezone.utc), model_display_name='lab02b-resnet-adm', model_resource_name='projects/588027604890/locations/europe-west3/models/1342622444770295808', version_aliases=['v1'], version_description='ResNet V2 101 — baseline, 224x224 input')
VersionInfo(version_id='2', version_create_time=DatetimeWithNanoseconds(2026, 3, 9, 14, 12, 1, 350777, tzinfo=datetime.timezone.utc), version_update_time=DatetimeWithNanoseconds(2026, 3, 9, 14, 12, 6, 218808, tzinfo=datetime.timezone.utc), model_display_name='lab02b-resnet-adm', model_resource_name='projects/588027604890/locations/europe-west3/models/1342622444770295808', version_aliases=['v2', 'default'], version_description='ResNet V2 50 — baseline, 128x128 input')


### 2.4 Retrieve the model version

In [13]:
##############################  TODO  ##############################
# List models filtering by display name to retrieve yours.
models = aip.Model.list(filter=f'display_name="{MODEL_DISPLAY_NAME}"') # TODO
####################################################################

print(f"Number of model resources: {len(models)}")
print(f"Default version ID: {models[0].version_id}")

model = models[0]

Number of model resources: 1
Default version ID: 2


**✏️ Question 2 — Default version**

Which version ID was returned? Why?

---
*Your answer:*

The version ID returned is 2 since the default model is the second version of the uploaded models (`is_default_version=True`)



---

---
## 3 · Endpoints

An **endpoint** is a managed HTTPS service that serves predictions from one or more deployed models. Creating an endpoint and deploying a model to it are two separate steps.

> 📖 **Docs:** [Deploy a model to an endpoint](https://cloud.google.com/vertex-ai/docs/predictions/deploy-model-api)

### 3.1 Create an endpoint
> 📖 **Docs:** [`Endpoint.create()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_create)

In [14]:
##############################  TODO  ##############################
# Create an endpoint with a display name that includes your_name.
endpoint = aip.Endpoint.create(
    display_name=f"lab02b-endpoint-{your_name}"  # TODO
)
####################################################################

print(f"✅ Endpoint created: {endpoint.resource_name}")

✅ Endpoint created: projects/588027604890/locations/europe-west3/endpoints/7129165224778661888


### 3.2 Deploy the model

Deploying a model provisions compute resources and starts a serving container. This can take **5–10 minutes**.

In [15]:
DEPLOY_COMPUTE = "n1-standard-4"

response = endpoint.deploy(
    model=model,
    deployed_model_display_name=f"lab02b-resnet-deployed-{your_name}",
    machine_type=DEPLOY_COMPUTE,
)

print(f"✅ Model deployed to endpoint.")

✅ Model deployed to endpoint.


> ⏳ **Deployment takes 5–10 minutes.** While you wait, answer the questions below.

**✏️ Question 3 — Endpoints & deployment**

a) What is the difference between an *endpoint* and a *deployed model*? Can an endpoint have multiple models?

b) You deployed using `n1-standard-4`. What factors would influence your choice of machine type in production? When would you use a GPU?


a) An endpoint is a set of resources and compute that hosts one or more deployments, each of which can contain a model or component for inference. An endpoint can be seen as an API interface where prediction requests are sent. A deployed model is a running model instance serving predictions.
An endpoint can route to one or many deployed models.

b) When choosing a machine type for production inference, you are balancing performance, latency, throughput, and cost. A GPU would be used when using large models that require some storage or when the inference should be quick.

### 3.3 Inspect the deployment

In [16]:
# Retrieve deployment details
deployed_info = endpoint.gca_resource.deployed_models[0]
print(f"Deployed model ID:   {deployed_info.id}")
print(f"Display name:        {deployed_info.display_name}")
print(f"Model resource name: {deployed_info.model}")

Deployed model ID:   5592329444124524544
Display name:        lab02b-resnet-deployed-adm
Model resource name: projects/588027604890/locations/europe-west3/models/1342622444770295808


---
## 4 · Manage Versions

### 4.1 Change the default version

You can reassign the `default` alias to a different version using `add_version_aliases()`.

> 📖 **Docs:** [`versioning_registry.add_version_aliases()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_versioning_registry)

In [17]:
##############################  TODO  ##############################
# Make version "1" the default again by assigning the "default" alias.
# Then verify by listing models and checking the version_id.

model_v2.versioning_registry.add_version_aliases(
    new_aliases=['default'],  # TODO
    version='1',        # TODO
)
####################################################################

# Verify the change
models = aip.Model.list(filter=f'display_name="{MODEL_DISPLAY_NAME}"')
print(f"Default version is now: {models[0].version_id}")

Default version is now: 1


### 4.2 Delete a version

You can delete a version using `delete_version()`.
> 📖 **Docs:** [`versioning_registry.delete_version()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_versioning_registry)

In [20]:
##############################  TODO  ##############################
# Delete the version v1.
# Hint: use the alias or version id.
model.versioning_registry.delete_version('1')  # TODO
####################################################################

# Verify deletion
versions = model.versioning_registry.list_versions()
print("Remaining versions:")
for v in versions:
    print(f"  {v}")

FailedPrecondition: 400 The Model version "projects/588027604890/locations/europe-west3/models/1342622444770295808@1" can't be deleted because it's a default version. Reassign the default alias and then delete it. If it's the only version, use DeleteModel instead.

**✏️ Question 4 — Deleting the default**

What happened when you tried to delete the default version of a model? Why is this restriction in place?

---
*Your answer:*

The default version cannot be deleted, to avoid deleting by mistake the most used version.



---

---
## 5 · Cleanup

Always clean up deployed resources. The order matters: undeploy first, then delete the endpoint, then delete the model.

In [21]:
# Undeploy all models from the endpoint
deployed_model_id = endpoint.gca_resource.deployed_models[0].id
endpoint.undeploy(deployed_model_id)
print(f"✅ Model undeployed (ID: {deployed_model_id})")

✅ Model undeployed (ID: 5592329444124524544)


In [22]:
# Delete the endpoint
endpoint.delete()
print("✅ Endpoint deleted.")

✅ Endpoint deleted.


In [23]:
# Delete the model (and all remaining versions)
model.delete()
print("✅ Model and all versions deleted.")

✅ Model and all versions deleted.


---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Prepare | Downloaded and saved two TFHub models to GCS | `tf_keras.Sequential`, `model.save()` |
| Upload | Registered v1 and v2 in the Model Registry | `aip.Model.upload()` |
| Version | Listed versions, changed the default, deleted a version | `versioning_registry` methods |
| Endpoint | Created an endpoint and deployed a model | `Endpoint.create()`, `endpoint.deploy()` |
| Inspect | Retrieved deployment metadata | `gca_resource.deployed_models` |
| Cleanup | Undeployed, deleted endpoint, deleted model | `undeploy()`, `delete()` |

**Next lab:** You will perform **hyperparameter tuning** on FashionMNIST and visualize results with **Vertex AI TensorBoard**.